# 12. 특징 추출

> 강의 실습 노트북 `12.feature.ipynb`의 셀을 그대로 정리했습니다.

강의 화면의 코드 순서와 파일명을 유지했습니다. 데이터 파일은 코드에 표시된 `./data` 또는 `../data` 상대 경로에 두세요.


## 라이브러리 준비


In [ ]:
import sys
import cv2
import numpy as np


## 그레이 영상 읽기


In [ ]:
src = cv2.imread('./data/lenna.bmp', cv2.IMREAD_GRAYSCALE)


## 읽기 검사


In [ ]:
if src is None:
    print('Image load failed!')
    sys.exit()


## Sobel 크기와 임계값


In [ ]:
dx = cv2.Sobel(src, cv2.CV_32F, 1, 0)
dy = cv2.Sobel(src, cv2.CV_32F, 0, 1)

mag = cv2.magnitude(dx, dy)
mag = np.clip(mag, 0, 255).astype(np.uint8)

dst = np.zeros(src.shape[:2], np.uint8)
_, dst = cv2.threshold(mag, 120, 255, cv2.THRESH_BINARY)

cv2.imshow('src', src)
cv2.imshow('mag', mag)
cv2.imshow('dst', dst)
cv2.waitKey()
cv2.destroyAllWindows()


## Canny 에지


In [ ]:
import cv2
import numpy as np
import sys

src = cv2.imread("./data/building.jpg", cv2.IMREAD_COLOR)
if src is None:
    print('Image load failed!')
    sys.exit()

dst = cv2.Canny(src, 50, 150)
cv2.imshow('src', src)
cv2.imshow('dst', dst)
cv2.waitKey()
cv2.destroyAllWindows()


## 확률적 Hough 선분


In [ ]:
import cv2
import numpy as np
import sys

src = cv2.imread("./data/building.jpg", cv2.IMREAD_COLOR)
if src is None:
    print('Image load failed!')
    sys.exit()

edges = cv2.Canny(src,50,150)

lines = cv2.HoughLinesP(
    image=edges,
    rho=1,
    theta=np.pi/180,
    threshold=160,
    minLineLength=70,
    maxLineGap=5
)
dst = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

if lines is not None:
    for i in range(lines.shape[0]):
        pt1 = (lines[i][0][0], lines[i][0][1])
        pt2 = (lines[i][0][2], lines[i][0][3])
        cv2.line(dst, pt1, pt2, (0, 0, 255), 2, cv2.LINE_AA)



cv2.imshow('src', src)
cv2.imshow('dst', dst)
cv2.waitKey()
cv2.destroyAllWindows()


## Hough 원


In [ ]:
import cv2
import sys

src = cv2.imread("./data/dial.jpg")

if src is None:
    print('Image load failed!')
    sys.exit()

gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)
bir = cv2.GaussianBlur(gray, (0, 0), 1.0)

def on_trackbar(pos):
    rmin = cv2.getTrackbarPos("minRadius", "img")
    rmax = cv2.getTrackbarPos("maxRadius", "img")
    th = cv2.getTrackbarPos("threshold", "img")


    circles = cv2.HoughCircles(
        image=bir,
        method=cv2.HOUGH_GRADIENT,
        dp=1,
        minDist=50,
        param1=120,
        param2=th,
        minRadius=rmin,
        maxRadius=rmax
    )

    dst = src.copy()

    if circles is not None:
        for i in range(circles.shape[1]):
            cx, cy, radius = circles[0][i]
            cv2.circle(
                dst,
                (int(cx),int(cy)),
                int(radius),
                (0,0,255),
                2,
                cv2.LINE_AA
            )
    cv2.imshow("img",dst)

cv2.imshow("img",src)
cv2.createTrackbar("minRadius", "img",1,100, on_trackbar)
cv2.createTrackbar("maxRadius", "img",1,150, on_trackbar)
cv2.createTrackbar("threshold", "img",1,100, on_trackbar)
cv2.setTrackbarPos("minRadius","img",10)
cv2.setTrackbarPos("maxRadius","img",80)
cv2.setTrackbarPos("threshold","img",40)
cv2.waitKey()

cv2.destroyAllWindows()


## 동전 원 검출


In [ ]:
import sys
import numpy as np
import cv2

src = cv2.imread('./data/coins1.jpg')

if src is None:
    print("Image open Failed!")
    sys.exit()

gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)
blr = cv2.GaussianBlur(gray, (0,0),1)

circles = cv2.HoughCircles(blr, cv2.HOUGH_GRADIENT, 1, 50,
                           param1=150,param2=40,minRadius=20,maxRadius=80)

sum_of_money=0
dst = src.copy()
if circles is not None:
    for i in range(circles.shape[1]):
        cx, cy, radius = circles[0][i]
        cx, cy, radius = list(map(int, [cx,cy,radius]))
        cv2.circle(dst, (cx,cy), radius, (0,0,255), 2, cv2.LINE_AA)

        x1 = int(cx - radius)
        y1 = int(cy - radius)
        x2 = int(cx + radius)
        y2 = int(cy + radius)

        radius = int(radius)

        crop = dst[y1:y2, x1:x2, :]
        ch, cw = crop.shape[:2]

        mask = np.zeros((ch, cw), np.uint8)
        cv2.circle(mask, (int(cw//2), int(ch//2)), radius, 255,-1)

        hsv = cv2.cvtColor(crop,cv2.COLOR_BGR2HSV)
        hue, _, _ = cv2.split(hsv)
        hue_shift = (hue +40) % 180
        mean_of_hue = cv2.mean(hue_shift, mask)[0]

        won = 100

        if mean_of_hue < 53:
            won = 10

        sum_of_money += won

        cv2.putText(crop,str(won), (20,50), cv2.FONT_HERSHEY_SIMPLEX,
                    0.75,(255,0,0),2,cv2.LINE_AA)
cv2.putText(dst, str(sum_of_money)+' won', (40,80),
            cv2.FONT_HERSHEY_DUPLEX, 2, (255,0,0),2,cv2.LINE_AA)

cv2.imshow('src',src)
cv2.imshow('dst',dst)
cv2.waitKey()

cv2.destroyAllWindows()
